In [ ]:
import os
import sys
from pathlib import Path

import torch
import torch.optim as optim
from torch.optim.lr_scheduler import LambdaLR

torch.manual_seed(8008135)

NOTEBOOK_DIR = Path.cwd()
CODE_DIR = NOTEBOOK_DIR.parent.parent

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

print("CODE_DIR:", CODE_DIR)
print("CODE_DIR contents:", os.listdir(CODE_DIR))

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print(f"Device set to {device}")

if device.type == "cuda":
    torch.set_float32_matmul_precision("high")

In [ ]:
from data.Datasets.Sudoku_DataLoader import get_loaders
from data.Datasets.Sudoku_DataLoader import collect_puzzles_set

sys.path.insert(0, os.path.join(CODE_DIR, "code"))
from BiLSTM_Model.BiLSTM_Sudoku import SudokuBiLSTM, BiLSTMConfig
from BiLSTM_Model.BiLSTM_Train import train_bilstm

from Utils.schedules import cosine_schedule_with_warmup_lr_lambda
from Utils.checkpointing import load_checkpoint
from Utils.visualization import show_sudoku_predictions, print_sudoku_comparison

In [ ]:
train_size = 2**21
test_size = 2**15
batch_size = 2**7

train_loader, val_loader = get_loaders(
    train_size=train_size,
    test_size=test_size,
    batch_size=batch_size,
)

In [ ]:
print("Collecting train puzzles...")
train_puzzles = collect_puzzles_set(train_loader)

print("Collecting val puzzles...")
val_puzzles = collect_puzzles_set(val_loader)

overlap = train_puzzles.intersection(val_puzzles)

print(f"\nTrain puzzles: {len(train_puzzles)}")
print(f"Val puzzles:   {len(val_puzzles)}")
print(f"Overlap:       {len(overlap)}")

if len(overlap) > 0:
    print("WARNING: Puzzle overlap detected!")
else:
    print("No puzzle overlap between train and validation sets.")

In [ ]:
model_config = BiLSTMConfig(
    vocab_size=10,
    num_classes=10,
    embedding_dim=512,
    hidden_size=560,
    num_layers=5,
    dropout=0.2
)

lr = 1e-4
min_lr_ratio = 0.1 # -> 1e-5
lr_warmup = 0.05
beta1 = 0.9
beta2 = 0.95
weight_decay = 0.1
num_epochs = 10 # gives same num_training_steps

checkpoint_dir = "checkpoints"

In [ ]:
model = SudokuBiLSTM(model_config).to(device)

print(
    "Number of trainable parameters:",
    f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}",
)

In [ ]:
optimizer = optim.AdamW(
    model.parameters(),
    lr=lr,
    betas=(beta1, beta2),
    weight_decay=weight_decay,
)

num_training_steps = len(train_loader) * num_epochs
num_warmup_steps = int(lr_warmup * num_training_steps)

scheduler = LambdaLR(
    optimizer,
    lr_lambda=lambda step: cosine_schedule_with_warmup_lr_lambda(
        step,
        num_warmup_steps=num_warmup_steps,
        num_training_steps=num_training_steps,
        min_ratio=min_lr_ratio
    ),
)

print("num_training_steps:", num_training_steps)
print("num_warmup_steps:", num_warmup_steps)

In [ ]:
model, best_metric, history = train_bilstm(
    model=model,
    train_loader=train_loader,
    optimizer=optimizer,
    device=device,
    scheduler=scheduler,
    num_epochs=num_epochs,
    checkpoint_dir=checkpoint_dir,
    checkpoint_every=5,
    validate_every=5,
    val_loader=val_loader,
    grad_clip=None,
    step_val_batches=1,
    step_val_every=8,
)
model.eval()

print("Best board accuracy used for checkpointing:", best_metric)

Same strange loss behavior as BERT, except Val Board Accuracy doesn't get as high

In [ ]:
import matplotlib.pyplot as plt

train_steps = history["step"]
train_loss = history["train_loss"]

val_steps = [
    s for s, v in zip(history["step"], history["val_loss"])
    if v is not None
]

val_loss = [
    v for v in history["val_loss"]
    if v is not None
]

plt.figure(figsize=(10, 5))
plt.plot(train_steps, train_loss, label="Train loss", linewidth=1)

if len(val_loss) > 0:
    plt.plot(val_steps, val_loss, label="Validation (subset)", linewidth=2)

plt.xlabel("Training step")
plt.ylabel("Loss")
plt.title("BiLSTM Loss vs Training Step")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
show_sudoku_predictions(
    model,
    val_loader,
    device,
)